In [17]:
import pandas as pd

HIST_PATH = ".../clean_football_data_v2.csv"
NEW_URL = "https://cfbtxt.com/data/schedule_2026.csv"
OUT_PATH = "../Data/football_games_with_2026.csv"

TEAM_FIX = {}   # fill in after the first run, e.g. {"San José State": "San Jose State"}
CONF_MAP = {"ACC": "acc", "B10": "big10", "B12": "big12", "SEC": "sec", "PAC": "pac12", "P12": "pac12",
            "MWC": "mwc", "AAC": "aac", "CUSA": "cusa", "MAC": "mac", "SBC": "sunbelt", "SUN": "sunbelt", "IND": "ind"}

hist = pd.read_csv(HIST_PATH)
new = pd.read_csv(NEW_URL)

done = new["completed"].astype(str).str.upper() == "TRUE"
tbd = new["time_tbd"].astype(str).str.upper() == "TRUE"
kick = pd.to_datetime(new["start_time_utc"], utc=True).dt.tz_convert("America/New_York")

rows = pd.DataFrame({
    "season": new["season"],
    "week": new["week"],
    "date": pd.to_datetime(new["date"]).dt.strftime("%m/%d/%Y"),
    "time_et": kick.dt.strftime("%I:%M %p").where(~tbd),
    "game_type": new["season_type"],
    "away": new["away_team"].replace(TEAM_FIX),
    "home": new["home_team"].replace(TEAM_FIX),
    "conf_away": new["away_conference"].map(CONF_MAP).fillna(new["away_conference"]),
    "conf_home": new["home_conference"].map(CONF_MAP).fillna(new["home_conference"]),
    "neutral": new["neutral_site"].astype(str).str.upper() == "TRUE",
    "score_away": new["away_points"].where(done),
    "score_home": new["home_points"].where(done),
    "home_win": (new["home_points"] > new["away_points"]).astype(int).where(done),
})

# --- Name checks: anything listed here won't connect to its old history ---
old_teams = set(hist["home"]) | set(hist["away"])
old_confs = set(hist["conf_home"].dropna()) | set(hist["conf_away"].dropna())
print("Team names NOT in your old data:", sorted((set(rows["home"]) | set(rows["away"])) - old_teams))
print("Conference codes NOT in your old data:", sorted((set(rows["conf_home"].dropna()) | set(rows["conf_away"].dropna())) - old_confs))

combined = pd.concat([hist[hist["season"] != 2026], rows], ignore_index=True)
combined.to_csv(OUT_PATH, index=False)
print(f"Added {done.sum()} finished + {(~done).sum()} upcoming 2026 games -> {OUT_PATH}")

Team names NOT in your old data: ['App State', 'Bethune-Cookman', 'Cal Poly', 'Central Connecticut', 'Elon', 'Florida International', 'Gardner-Webb', 'Grambling', "Hawai'i", 'Long Island University', 'Louisiana', 'McNeese', 'Mercyhurst', 'Miami', 'Nicholls', 'SE Louisiana', 'Sam Houston', 'San José State', 'South Florida', 'Southeast Missouri State', 'Stonehill', 'The Citadel', 'UAlbany', 'UC Davis', 'UConn', 'UL Monroe', 'UT Martin', 'UT Rio Grande Valley', 'VMI', 'West Georgia']
Conference codes NOT in your old data: ['Big Sky', 'Coastal Athletic', 'FCS Independents', 'MEAC', 'MVFC', 'NEC', 'OVC', 'Patriot', 'SWAC', 'Southern', 'Southland', 'UAC', 'sunbelt']
Added 391 finished + 497 upcoming 2026 games -> ../Data/football_games_with_2026.csv


In [18]:
for word in ["Appalachian", "App", "FIU", "Florida Int", "Haw", "Louisiana", "Lafayette", "Miami",
             "San Jos", "USF", "South Florida", "Conn", "Monroe", "ULM", "Sam Houston"]:
    print(f"{word:15} ->", sorted(n for n in old_teams if word.lower() in n.lower()))

Appalachian     -> ['Appalachian State']
App             -> ['Appalachian State']
FIU             -> ['FIU']
Florida Int     -> []
Haw             -> ['Hawaii']
Louisiana       -> ['Louisiana Tech', 'Southeastern Louisiana']
Lafayette       -> ['Lafayette', 'UL-Lafayette']
Miami           -> ['Miami (FL)', 'Miami (OH)']
San Jos         -> ['San Jose State']
USF             -> ['USF']
South Florida   -> []
Conn            -> ['Central Conn. State', 'Connecticut']
Monroe          -> ['UL-Monroe']
ULM             -> []
Sam Houston     -> ['Sam Houston State']


In [19]:
TEAM_FIX = {
    "App State": "Appalachian State",
    "Florida International": "FIU",
    "Hawai'i": "Hawaii",
    "Louisiana": "UL-Lafayette",
    "Miami": "Miami (FL)",
    "San José State": "San Jose State",
    "South Florida": "USF",
    "UConn": "Connecticut",
    "UL Monroe": "UL-Monroe",
    "Sam Houston": "Sam Houston State",
    "SE Louisiana": "Southeastern Louisiana",
    "Central Connecticut": "Central Conn. State",
}